# Welcome to the 3DVar tutorial
***
In this tutorial we will explore the basic features of 3DVar, using a so-called quasi-geostrophic model. The model mimics the evolution of large-scale features, such as high and low pressure cells in the atmosphere. In the following we first provide a short description of  the 3DVar problem and of the quasi-geostrophic model and what it represents, followed by the data-assimilation experiments.

**Be sure to run 0.qg_tutorial_start.ipynb before this tutorial**

## Introduction to 3DVar

*For those familiar with the 3DVar data-assimilation method this section can be skipped.* <br>
3DVar is a data-assimilation method that solves the data-assimilation problem at a given point in time. The basic idea is that we have a first guess of the atmospheric state from a model forecast $x_b$, and want to update this state with a set of observations $y$ obtained at this time. These observations are from the unknown true atmospheric state, and the measurement process is described by <br>
<center>$y = H(x_{true}) + \epsilon_{true}$ </center> <br>
in which $H(..)$ is the so-called observation operator that maps a state $x$ to observation space. The $\epsilon_{true}$ is the observation error. This error is due to the measurement process itself: every measurement will have an error related to the precision of the measurement. There is also another part to this error, called the representation error, that we will not discuss here as it is not essential for understanding 3DVar. <br><br>
The 3DVar tries to find the state that is both close to our first guess $x_b$ and close to the observations $y$. It seems natural to measure closeness in terms of the uncertainty in each of them. If the first guess is less certain, the solution doesn't have to be that close to the prior than if it had very small uncertainty. This suggests to set up a variational problem in which we minimize a so-called cost function J(x), defined as <br>
<center>$J(x) = \frac{(x-x_b)^2}{\sigma_b^2} + \frac{(y-x)^2}{\sigma_y^2}$ </center> <br>
in which $\sigma_b$ is the standard deviation of the error in $x_b$, and $\sigma_y$ is the standard deviation of the error in $y$. If, for instance, $\sigma_b$ is much larger than $\sigma_y$ the first term will become important in the cost function for large variations of $x$ from $x_b$, while the second term becomes important for much smaller variations of $x$ from $y$. This shows that this cost function makes sense. (This cost function might seem ad hoc, but it can be derived from Bayes Theorem.) The minimum of this cost function is found by setting its gradient to zero. In this simple problem we find for the gradient:<br>
<center>$\frac{d J(x)}{dx} = 2\frac{(x-x_b)}{\sigma_b^2} + 2\frac{(y-x)}{\sigma_y^2}$ </center> <br>
which we can solve for $x=x_a$, the state at which the gradient is zero, as<br>
<center>$x_a=x_b +\frac{\sigma_b^2}{\sigma_b^2+\sigma_y^2}(y-x_b)$ </center> <br>
We see that the analysis state is a linear combination of the first guess and the observation. To see this more clearly we can rewrite the equation as <br>
<center>$x_a=\frac{\sigma_y^2}{\sigma_b^2+\sigma_y^2}x_b +\frac{\sigma_b^2}{\sigma_b^2+\sigma_y^2}y$ </center> <br>
where the analysis is expressed as a weighted average of first guess and observation. When the observation error is much larger than the model error the weight on the first guess $x_b$ is much larger than the weight on the observation, so the analysis will be close to the first guess. This is exactly what we want the data assimilation to do: the analysis should be closest to the quantity with the smallest error.

The 3DVar method allows for extending the cost function in two ways, the problem can be multi-dimensional and the observation operator can be nonlinear. This leads to a cost function

<center>$J(x) = (x-x_b)^TB^{-1}(x-x_b) + \left(y-H(x)\right)^T R^{-1}\left(y-H(x)\right)$ </center> <br>
The deviation of $x$ from $x_b$ is weighted by the first-guess error covariance $B$, and deviation of $x$ from $y$ is weighted by the observation error covariance. The gradient of the cost function is given by <br>
<center>$\frac{d J(x)}{dx} = 2B^{-1}(x-x_b) - 2H^T R^{-1}\left(y-H(x)\right)$ </center> <br>
Note the similarity with and the difference from the scalar case above, especially the error variances there and the covariance matrices here. If $H(..)$ is a linear function we can find an analytical expression for the state $x_a$ for which the gradient is zero. Doing that and collecting all appearances of $x-x_b$ on the left hand side and then subtract $H^TR^{-1}Hx_b$ from both sides of the equation results in <br>
<center>$\left(B^{-1}+H^TR^{-1}H \right) (x_a-x_b) =  H^T R^{-1}(y-Hx_b)$ </center> <br><br>
The solution is now obtained as:<br>
<center>$x_a = x_b + \left(B^{-1}+H^TR^{-1}H \right)^{-1}H^T R^{-1}(y-Hx_b)$ </center> <br>
which some of you might recognize as the Kalman filter solution in precision form. Indeed, 3DVar and the Kalman Filter give the same solution as the 3DVar when $H(..)$ is linear. We can rewrite this equation as<br>
<center>$x_a = \left(B^{-1}+H^TR^{-1}H \right)^{-1} B^{-1}x_b + \left(B^{-1}+H^TR^{-1}H \right)^{-1}H^T R^{-1}y$ </center> <br>
where we can immediately see how the first-guess and observation covariances determine the analysis state. When the observation covariance is large, so the observations are inaccurate, the first term dominates and the analysis is close to the first guess. If, on the other hand, the observations are very accurate the second term dominates and the analysis will be closer to the observations.<br><br>

To make the connection with the scalar case we can use some matrix identities to rewrite the analysis equation as the standard covariance form of the Kalman filter<br>

<center>$x_a = x_b + BH^T\left(HBH^T+R \right)^{-1}(y-Hx_b)$ </center> <br>
We can multiply this equation with H and rewrite it as<br>

<center>$Hx_a = R\left(HBH^T+R \right)^{-1}x_b + BH^T\left(HBH^T+R \right)^{-1}y$ </center> <br>
and a direct identification with the scalar case can be made in terms of error variances in the scalar case and error covariances in the multi-dimensional case. The 3DVar solution is a weighted average between the first guess in observation space and the observations, with the weight determined by the covariances of first guess and observation errors.<br><br>

Going back to Kalman filter equation in covariance form, which, as mentions is also the solution of the 3DVar problem, the importance of the $B$ matrix becomes apparent. The so-called innovation $y-Hx_b$, so the difference between the observation and the model forecast in observation space, is first multiplied by $\left(HBH^T+R \right)^{-1}$, and the resulting vector is still in observation space. The transition to state space is made via the multiplication by $BH^T$. Assume that we only have one observation of, say, pressure at a certain point in space. That observation will not only update the pressure at that point, but it will influence many other variables too, depending on the covariance structure of the B matrix.<br><br>

When $H(..)$ is nonlinear we cannot write down an explicit solution, and instead have to use iterative methods to solve this problem. This makes the 3DVar, which is an iterative method, more general than the Kalman Filter.<br><br>

However, even when $H(..)$ is linear the 3DVar has advantages in high-dimensional problems such as atmospheric or oceanic data assimilation. When we look at the solution we see that we need the inverse of the background error covariance B, which is a huge matrix. In fact, it is typically so large that we cannot even store it on a computer, let alone determine its inverse. For this reason a 3DVar is typically implemented as follows. First, a control-variable transformation is used, in which a new variable $u$ is defined via $x-x_b = B u$. (Note that it is also possible to use $B^{1/2}$ in the transform, but JEDI uses $B$.) Then the problem<br><br>
<center>$\left(B^{-1}+H^TR^{-1}H \right) (x-x_b) =  H^T R^{-1}(y-Hx_b)$ </center> <br>
becomes<br>
<center>$\left(I+H^TR^{-1}HB \right) u =  H^T R^{-1}(y-Hx_b)$ </center> <br>
which is of the form $Au = b$, and this equation is solved iteratively. The matrix $B$ is not stored as a matrix, but instead as a piece of code that takes as input a vector $u$, and generates as output $B u$.<br><br>

With this we are at the end of this short introduction of 3DVar. Next we very briefly describe the quasi-geostrophic model, before we dive into the experiments with JEDI.

In [34]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

Now, try to run the following code:

In [35]:
ls -RCd --color $JEDI_BIN/qg*            # list the files in this directory
ls -RCd --color $JEDI_EDU/qg3Dvar/*/*


/home/nonroot/build/bin/qg_4dvar.x
/home/nonroot/build/bin/qg_addincrement.x
/home/nonroot/build/bin/qg_convertincrement.x
/home/nonroot/build/bin/qg_convertstate.x
/home/nonroot/build/bin/qg_dfi.x
/home/nonroot/build/bin/qg_diffstates.x
/home/nonroot/build/bin/qg_dirac.x
/home/nonroot/build/bin/qg_eda.x
/home/nonroot/build/bin/qg_ens_forecast.x
/home/nonroot/build/bin/qg_ens_hofx.x
/home/nonroot/build/bin/qg_ens_recenter.x
/home/nonroot/build/bin/qg_ens_variance.x
/home/nonroot/build/bin/qg_forecast.x
/home/nonroot/build/bin/qg_gen_ens_pert_B.x
/home/nonroot/build/bin/qg_hofx.x
/home/nonroot/build/bin/qg_hofx3d.x
/home/nonroot/build/bin/qg_hybridgain.x
/home/nonroot/build/bin/qg_letkf.x
/home/nonroot/build/bin/qg_rtpp.x
/home/nonroot/build/bin/qg_staticbinit.x
/home/nonroot/shared/EDU/qg3Dvar/output/exp_default
/home/nonroot/shared/EDU/qg3Dvar/output/exp_hor_len
/home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs
/home/nonroot/shared/EDU/qg3Dvar/output/exp_std_BR
/home/nonroot/shared

<br>
Look at the directory structure you just displayed: this is where you will need to navigate to find plots, yamls, analysis et background files... 

The files you will change are the **yaml files** (see where they are in the structure above). When you run jedi it will create output as indicated. Note that each experiment has a *bg* directory with the background files, an *obs* directory with the observation files and a *da* directory with the analysis files. These files are generated by jedi and can  also be vizualized using python, as explained later.


Now you are all ready to go!

## Overview of the experiments

In the following you will conduct several experiments. Most will be experiments in which you will assimilate one observation. This will allow you to better understand the JEDI system, but you will especially appreciate how 3DVar works, specifically the role of the covariances matrices for observations and the background. This understanding is a very good introduction to most present-day data assimilation methods, as the way the covariances work is very similar in 3DVar, in Ensemble Kalman Filters, in 4DVar, and in hybrid methods. The experiments with one observation are as follows:

| Description | Background yaml file   | Observation yaml file | Data assimilation file |
| --- | --- | --- | --- |
| exp_default | genenspert_B_default.yaml | makeobs3d_oneobs.yaml | 3dvar_oneobs_default.yaml
| exp_hor_len | genenspert_B_hor_len.yaml | makeobs3d_oneobs.yaml | 3dvar_oneobs_hor_len.yaml
| exp_ver_len | genenspert_B_ver_len.yaml | makeobs3d_oneobs.yaml | 3dvar_oneobs_ver_len.yaml
| exp_mult_obs | genenspert_B_default.yaml | makeobs3d_mult_obs.yaml | 3dvar_mult_obs.yaml

Our last experiment with 3DVar will be a more realistic case with 100 randomly positioned observations.

# Experiment 1: 3dvar with one observation
***

### Step 1: Gather Truth Simulation

We will copy over the truth simulation previously generated to the qg3Dvar directory (please run through **`0.qg_tutorial_start.ipynb`** first if you have not done so yet)

In [ ]:
cd $JEDI_EDU/
cp ./qgstart/output/truth/truth*nc ./qg3Dvar/output/truth

In [ ]:
ls $JEDI_EDU/qg3Dvar/output/truth

### Steps 2-3: Background State and Observations

In this step we will copy over the generated background state and single observation that was previously generated from the **`0.qg_tutorial_start.ipynb`** tutorial

In [ ]:
cd $JEDI_EDU
cp  ./qgstart/output/bg/bkgd*nc  ./qg3Dvar/output/exp_default/bg
cp  ./qgstart/output/obs/truth*nc ./qg3Dvar/output/exp_default/obs

### Step 4: Run the 3Dvar!

We are now in a position to perform the data assimilation, in this case a 3dvar. Have a look at `3dvar_oneobs_default.yaml` and see if you can understand every line:

```yaml
cost function:
  cost type: 3D-Var  # Set to activate 3dvar routines in JEDI 
  window begin: 2009-12-30T21:00:00Z
  window length: PT6H
  analysis variables: [x]  # DA analysis uses streamfunction (x); analysis corrections in other variables are diagnosed
  geometry:
    nx: 40
    ny: 20
    depths: [4000.0, 6000.0]
  background:
    date: 2009-12-31T00:00:00Z
    filename: qg3Dvar/output/exp_default/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
  background error:                # Define the background error covariance settings 
    covariance model: QgError      # Defines the covariance model (the built-in QgError model within JEDI)
    horizontal_length_scale: 2.2e6 # Defines the horizontal length scale of error covariances in meters 
    maximum_condition_number: 1.0e6
    standard_deviation: 1.8e7      # Defines the standard deviation of error in streamfunction (m2/s)
    vertical_length_scale: 15000.0 # Defines the vertical length scale of error covariances in meters
  observations:
   observers:
    - obs error:                   # obs error info contained in observation file (truth_oneob.obs3d.nc)
        covariance model: diagonal # which was defined when we made obs using makeobs3d_oneobs.yaml (see 0.qg_tutorial_start)
      obs operator:
        obs type: Stream
      obs space:
        obsdatain:
          engine:
            obsfile: qg3Dvar/output/exp_default/obs/truth_oneob.obs3d.nc # Input observation file
        obsdataout:
          engine:
            obsfile: qg3Dvar/output/exp_default/obs/3dvar.obs3d.nc # Output (diagnostic) observation file with analysis
        obs type: Stream
variational:
  minimizer:
    algorithm: DRIPCG                  # The minimizer is a variant of the conjugate gradient algorithm
  iterations:
  - diagnostics:                       # First outer loop
      departures: ombg                 # Generate the observations - background departures
    gradient norm reduction: 1.0e-10   # Stop the minimization when either the gradient norm is this size
    ninner: 10                         # or when the number of inner loops exceeds 10
    geometry:                          # Use reduced model resolution for first outer loop
      nx: 20
      ny: 10
      depths: [4000.0, 6000.0]
    test: on
    online diagnostics:                 
      write increment: true            # Output increments of first outer loop
      increment:
        datadir: qg3Dvar/output/exp_default/da
        date: 2009-12-31T00:00:00Z
        exp: 3dvar.iter1
        type: in
  - diagnostics:                       # Second outer loop
      departures: ombg
    gradient norm reduction: 1.0e-10
    ninner: 10
    geometry:
      nx: 40                           # Use full model resolution for second outer loop
      ny: 20
      depths: [4000.0, 6000.0]
    test: on
    online diagnostics:
      write increment: true             # Output increments of first outer loop
      increment:
        datadir: qg3Dvar/output/exp_default/da
        date: 2009-12-31T00:00:00Z
        exp: 3dvar.iter2
        type: in
final:
  diagnostics:
    departures: oman                   # After minimization generate observations - analysis departures
  increment:                           # Save the final analysis increments to an output file
    geometry:
      nx: 40
      ny: 20
      depths: [4000.0, 6000.0]
    output:
      datadir: qg3Dvar/output/exp_default/da
      date: 2009-12-31T00:00:00Z
      exp: 3dvar.increment
      filename: 3dvar.increment
      type: in
      analysis variables: [x]
output:                                # Save the final full analysis to an output file
  datadir: qg3Dvar/output/exp_default/da
  exp: 3dvar
  frequency: PT6H
  type: an

```

Run this code via

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg3Dvar/yamls/3dvar_oneobs_default.yaml 

**You have done it! Your first ever data-assimilation run with JEDI!**

There will be four files in `/home/nonroot/shared/EDU/qg3Dvar/output/exp_default/da`. List them with the command below, do you know what each file represents?

In [ ]:
ls -lh $JEDI_EDU/qg3Dvar/output/exp_default/da

We will now plot the results using the python plotting script. The first plot is a plot of the increments from the data assimilation. 
Run:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_default/da/3dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/exp_default/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qg3Dvar/output/exp_default/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 30000000.0 \
        --output $JEDI_EDU/qg3Dvar/output/exp_default/plots/3dvar_increment --title "analysis increment"

To view these plots, navigate in the directory tree on the left to **shared->EDU->qg3Dvar->output**. Each experiment directory (e.g. exp_default) contains a `plots` diretory. These plots are located in **exp_default->plots**. They can be opened by double-clicking. They can also be "downloaded" to your local machine (e.g. Desktop or Documents directory) by right clicking and clicking on the Download button.

Your plots should look very much like the following. <u> Note the observation location, marked by the 'x', at the center of the increments.</u><br>
<center> <img src="images/3dvar_increment_x_diff.jpg"/> </center>
<br>
Let's see if this figure make sense, and learn a lot about the working of 3DVar in the process. The figure shows the increment in the observed variable, the stream function. But wait, we only observed one grid point in the upper layer, but we see analysis increments *in a much larger area*. To understand this, remember the 3DVar update equation, which for a linear observation operator $H$ is equal to that of the Kalman filter:<br>

<center>$x_a = x_b + BH^T\left(HBH^T+R \right)^{-1}(y-Hx_b)$</center><br>

Since there is only one observation $R$ is a scalar, the observation error variance. This means that $HBH^T$ also has to be a scalar (and it is indeed). Remember hat $H$ maps the state vector in observation space, so the full 1600 (40 by 20 by 2) dimensional state vector is mapped to a 1-dimensional scalar. This means that $H$ is a matrix with 1 row and 1600 columns, and $HBH^T$ is just a scalar.<br><br>
Hence, the spreading of information over the larger domain has to come from $BH^T$, which is a matrix of size 1600 by 1. Indeed, it maps the scaled innovation $\left(HBH^T+R \right)^{-1}(y-Hx_b)$ back to state space. The covariances between grid points in the prior error covariance $B$ spread this information out in state space. Remember that the spatial length scale in the $B$ matrix is 2200 km, and the grid spacing is 1000km, so 2.2 grid points. The figure shows that the spread of information is indeed about twice as large as the correlation length scale specified for the B matrix, as expected for a Gaussian correlation structure.<br><br>
This raises the question is we could set this correlation length scale much larger, for instance to get an update over the whole domain just from one observation. Pushing this a bit, one might argue that satellites are not needed, we just need a set of well chosen surface observations and large length scales in the $B$ matrix. However, that thinking is incorrect. The $B$ matrix should contain the length scales of the physical processes. The relevant processes are the high- and low pressure systems, as seen in the true state, one of the first figures in this tutorial above. Indeed, the length scales of the increment have the length scales seen in that figure, which means that we have set the length scales in the $B$ matrix correctly.<br><br>
The physical reason for this length scale is that a pressure perturbation at a certain grid point will be physically related to other pressure perturbations around it, and the typical length scale of these relations is the length scale that should be used in the $B$ matrix.<br><br>

A question: back to the figure, we only observed a grid point in the upper layer, but there is an increment in the lower layer too. Can you figure out why, perhaps from the yaml file used to generate the 3Dvar solution, or in the yaml file used to generate the background state?<br><br>

Indeed, it is the **vertical_length_scale**, which is set to 15000 m. Given the layer thicknesses of 4000 and 6000 m, the two layers are highly correlated, and an increment in the upper layer will be quite similar in the lower layer.<br><br>

As a final thing we will look at the **magnitude** of the increment. At the observation location we can write, because we have only one observation:
<br>
<center> $x_a = x_b + b\left(b+r \right)^{-1}(y-x_b)$ </center> <br>

in which $b$ is the background error variance at the observed location, $r$ is the variance of the observation error, and we removed the $H$ because we are observing the stream function directly at this grid point, so $H=1$. Using this equation and the values for $b$ from the background error standard deviation ($1.8e7 m^2/s$, so $b = 3.24e14$) and for $r$ from the observation error standard deviation ($4.0e6\;m^2/s$, so $r=1.6e13$), and your values for the observation and the background state at that grid point (estimate the latter from the background error and the truth value in the observation file), check that the value of the stream function increment makes sense.<br><br>

Next, recall the value of the observation, which we already printed to a text file from the 'start' tutorial:



We need the value of the observation, and we also need the background value xb at the observation location. To extract this information, we can run the `qg_hofx3d.x` program as we did in the qgstart tutorial, only this time we have the observation already generated from the truth (rather than making the observation), and in this case we only want to compute the hofx value at the observation location. The yaml file **hofx_default.yaml** shows these settings:

```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: 2009-12-31T00:00:00Z
  filename: qg3Dvar/output/exp_default/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
window begin: 2009-12-30T21:00:00Z
window length: PT6H
observations:
  get values:
    variable change:
      input variables: []
      output variables: []
  observers:
  - obs operator:
      obs type: Stream                             # The observation type is the stream function
    obs space:
      obsdatain: 
        engine:
          obsfile: qg3Dvar/output/exp_default/obs/truth_oneob.obs3d.nc
      obsdataout:
        engine:
          obsfile: qg3Dvar/output/exp_default/obs/hofx_bkgd_oneob.obs3d.nc
      obs type: Stream
    obs operator:
      obs type: Stream
    get values:
      interpolation type: default_1
```

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qg3Dvar/yamls/hofx_default.yaml

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qg3Dvar/output/exp_default/obs/hofx_bkgd_oneob.obs3d.nc \
        --output $JEDI_EDU/qg3Dvar/output/exp_default/plots/hofx_bkgd_oneob
cat $JEDI_EDU/qg3Dvar/output/exp_default/plots/hofx_bkgd_oneob.txt

So for our experiment, the observation value $y \approx -2.799e8$ and $x_b \approx -2.525e8$  leading to an estimate analysis increment for the stream function of $0.95*(-2.799e8-(-2.525e8)) = -2.6e7$, which is consistent with the magnitude of the 3dvar analysis increments as shown in the figure above at the observation location
<br><br>
We can also look at how derived variables are affected by these increments in stream function. The next two figures show the zonal and meridional velocity increments.<br>

<center> <img src="images/3dvar_increment_u_diff.jpg"/> <img src="images/3dvar_increment_v_diff.jpg"/> </center>
<br><br>
Remember that they are related to the streamfunction through<br>

<center>$u = -\frac{\partial \psi}{\partial y} \;\;\;\;\; v = \frac{\partial \psi}{\partial x}$</center> <br>

Check that signs and orders of magnitude are indeed as you would expect.

While these increments are directly derived from the increments in the stream function, this will be more interesting in a numerical weather prediction model. There the pressure (corresponding to the stream function in a QG model) and the velocities are not necessarily in geostrophic balance; we do have ageostrophic velocities in the real atmosphere, and in our more complete numerical models. Still, the velocity increments will be similar to a pressure observation. Why do you think that is?

The answer is that we know that the winds will be in geostrophic balance to a very large extend, so we will build an approximate geostrophic relation into the $B$ matrix. How this is done is beyond this tutorial, but this is standard practice in 3DVar, and has been for many decades.

The final figure is the potential vorticity increment:<br>
<center> <img src="images/3dvar_increment_q_diff.jpg"/> </center>
<br><br>

Check if this makes sense to you.

After you have done that, note the anti-cyclonic circulation related to the positive mass anomaly from the stream function increment field.

This concludes the first 3DVar data-assimilation experiment. Please check that you:

1. Understand how to generate a truth run, the background field, observations, and a data assimilation run in JEDI,
2. Understand how the background error covariance matrix determines how observation information is spread around over the model state
3. Get an initial idea of how error magnitudes influence the increments. More about that in the next experiments!

In the next experiments we will change background and observation characteristics to get a better feel for the 3DVar and for the JEDI system.

# Experiment 2: Change the horizontal length scale of the background error
***

In the experiment **exp_hor_len**, we decrease the horizontal length scale of the background error covariance, so instead of using the same as in **exp_default** `horizontal_length_scale: 2.2e6` we will use `horizontal_length_scale: 1.1e6`. We will follow the same 4-step procedure as before.<br>
### Step 1: Truth
The truth should be exactly the same, we can skip step 1.<br>

### Step 2-3: Background & Observatuions
As in the first experiment, we use observations and background generated from the **`0.qg_tutorial_start.ipynb`** tutorial. Make a copy of the truth and background files to the experiment directory **(exp_hor_len)**

In [ ]:
cd $JEDI_EDU
cp  ./qgstart/output/bg/bkgd*nc  ./qg3Dvar/output/exp_hor_len/bg
ls $JEDI_EDU/qg3Dvar/output/exp_hor_len/obs

cp  ./qgstart/output/obs/truth*nc ./qg3Dvar/output/exp_hor_len/obs
ls $JEDI_EDU/qg3Dvar/output/exp_hor_len/obs

### Step 4: Run the data assimilation:
Copy and paste the yaml file `3dvar_oneobs_default.yaml`:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_default.yaml $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_hor_len.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

This new yaml file is located at `/home/nonroot/shared/EDU/qg3Dvar/yamls`. You can open and edit this file within this notebook by navigating to that directory using the tree to the left, and then double-clicking the yaml file to open it. Make your edits, then click the save button at the top left to save changes (or CTRL+s/⌘+s on windows/mac as keyboard shortcut) <br><br>
**You need to open `3dvar_oneobs_hor_len.yaml` and modify the following way:**
- Change`horizontal_length_scale` under the `background error` section to 1.1e6.
- Replace <u>all</u> the occurences of `exp_default` to `exp_hor_len`: this ensures we are aren't overwriting any previous results. There should be **seven** locations in the yaml that need to be updated here (***Tip**: use `Command+F` on Mac or `Ctrl+F` on Windows to open a find window, and find and replace all at once!*)

Once those changes are made, the next step is to run the experiment!

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg3Dvar/yamls/3dvar_oneobs_hor_len.yaml

If your run completed without errors, check that results (4 files) are produced in your `exp_hor_len/da` directory

Note: If you have errors, double-check that you have copied the `bg` and `obs` files correctly in steps 2-3, and the paths for inputs and outputs in the `3dvar_oneobs_hor_len.yaml` file are correct.

In [ ]:
ls $JEDI_EDU/qg3Dvar/output/exp_hor_len/da

<br>Before you look at the results, take a moment to think and form an idea of what you expect to see. After that, have a look at the results via the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_hor_len/da/3dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/exp_hor_len/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qg3Dvar/output/exp_hor_len/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 30000000.0 --output $JEDI_EDU/qg3Dvar/output/exp_hor_len/plots/3dvar_hor_len_increment --title "analysis increment"

<br>To view these plots, navigate in the directory tree on the left to **shared->EDU->qg3Dvar->output->exp_hor_len->plots**<br>
<br>Study the results and try to answer the following questions:

- Why is the increment area smaller than in the default experiment? Is this what you expected?
- Do the velocities have the same magnitude as in the default experiment? Why, or why not?
- Try to make sure you understand 'everything' before you move on!

This experiment should have:
- Increased your skills in using and changing yaml files
- Given you a better understanding of the influence of the horizontal length scales used in the $B$ matrix.

# Experiment 3: Change the vertical length scale of the background error.
***

In this experiment, we decrease the vertical length scale of the background error covariance, so instead of using
in `vertical_length_scale: 15000.0` like in **exp_default** we will use `vertical_length_scale: 5000.0` in **exp_ver_len**.

### Step 1: Truth
The truth should be exactly the same. So, we can skip step 1.

### Steps 2-3: Background & Observations

As in the first experiment, we use observations and background generated from the **0.qg_tutorial_start.ipynb** tutorial. Make a copy of the observation and background files to the experiment directory **(exp_ver_len)**. Double-check the files were copied correctly via the 'ls' commands.


In [ ]:
cd $JEDI_EDU
cp  ./qgstart/output/bg/bkgd*nc  ./qg3Dvar/output/exp_ver_len/bg
ls $JEDI_EDU/qg3Dvar/output/exp_ver_len/bg

cp  ./qgstart/output/obs/truth*nc ./qg3Dvar/output/exp_ver_len/obs
ls $JEDI_EDU/qg3Dvar/output/exp_ver_len/obs

The output file will be `/home/nonroot/shared/EDU/qg3Dvar/output/exp_ver_len/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc`

### Step 4: Run the data assimilation
Copy and paste the yaml file `3dvar_oneobs_default.yaml` to `3dvar_oneobs_ver_len.yaml`:

In [ ]:
cp $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_default.yaml $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_ver_len.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

You need to open `3dvar_oneobs_ver_len.yaml` and modify several values:
- Reduce `vertical_length_scale` under the `background error` section to **5000.0**.
- Replace all the occurences of `exp_default` with `exp_ver_len`: this ensures we do not overwrite any previous results. There should be **seven** total occurrences of this change to make in the yaml file

Once this is completed, we can run this experiment via:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg3Dvar/yamls/3dvar_oneobs_ver_len.yaml

If your run completed without errors, check that results (4 files) are produced in your `exp_ver_len/da` directory. 

Note: If you have errors, double-check that you have copied the `bg` and `obs` files correctly in steps 2-3, and the paths for inputs and outputs in the `3dvar_oneobs_ver_len.yaml` file are correct.

In [ ]:
ls $JEDI_EDU/qg3Dvar/output/exp_ver_len/da

<br>
What do you expect to see in the increments? Compare your ideas with the results using the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_ver_len/da/3dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/exp_ver_len/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsLocations $JEDI_EDU/qg3Dvar/output/exp_ver_len/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 30000000.0 --output $JEDI_EDU/qg3Dvar/output/exp_ver_len/plots/3dvar_ver_len_increment --title "analysis increment"

<br>To view these plots, navigate in the directory tree on the left to **shared->EDU->qg3Dvar->output->exp_ver_len->plots**<br><br>
Check the differences between these plots and the default experiment.
- Is the increment in the upper layer the same?
- Why is the increment in the lower layer smaller?

This experiment should have
- Increased your skills in using and changing yaml files
- Given you a better understanding of the influence of the horizontal length scales used in the $B$ matrix.

# Experiment 4: Assimilate multiple observations
***

We now embark on a more realistic situation in which 100 observations are assimilated all at once.

### Step 1: Truth
The truth should be exactly the same. So, we can skip step 1.

### Step 2: Background
We can just copy the file from the exp_default experiment:

In [36]:
cp $JEDI_EDU/qg3Dvar/output/exp_default/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc $JEDI_EDU/qg3Dvar/output/exp_mult_obs/bg/
ls $JEDI_EDU/qg3Dvar/output/exp_mult_obs/bg/

bkgd.fc.2009-12-30T00:00:00Z.P1D.nc


### Step 3: Observations
Copy the previous, randomly-generated 100 observations (truth_100obc.obs3d.nc) from the **`0.qg_tutorial_start.ipynb`** tutorial

In [37]:
cd $JEDI_EDU
cp ./qgstart/output/obs/truth_100obs.obs3d.nc ./qg3Dvar/output/exp_mult_obs/obs
ls $JEDI_EDU/qg3Dvar/output/exp_mult_obs/obs

3dvar.obs3d.nc  truth_100obs.obs3d.nc


### Step 4: Run the data assimilation

Copy and paste the yaml file 3dvar_oneobs_default.yaml:

In [38]:
cp $JEDI_EDU/qg3Dvar/yamls/3dvar_oneobs_default.yaml $JEDI_EDU/qg3Dvar/yamls/3dvar_mult_obs.yaml
ls $JEDI_EDU/qg3Dvar/yamls/

3dvar_mult_obs.yaml        genenspert_B_default.yaml  hofx_mult_obs.yaml
3dvar_oneobs_default.yaml  genenspert_B_hor_len.yaml  makeobs3d_mult_obs.yaml
3dvar_oneobs_hor_len.yaml  generate_truth.yaml        makeobs3d_oneobs.yaml
3dvar_oneobs_ver_len.yaml  hofx_default.yaml


You need to open `3dvar_mult_obs.yaml` and modify several values:
- replace all the occurences of `exp_default` with `exp_mult_obs`: this ensures we aren't overwriting any previous results. This should be done in **seven** locations within the yaml file.
- rename `truth_oneob.obs3d.nc` to `truth_100obs.obs3d.nc` so we assimilate with the generated 100 observations

and run the 3Dvar:

In [39]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qg3Dvar/yamls/3dvar_mult_obs.yaml

Configuration input file is: ./qg3Dvar/yamls/3dvar_mult_obs.yaml
OOPS Starting 2026-02-12 18:10:45 (UTC+0000)
Full configuration is:YAMLConfiguration[path=./qg3Dvar/yamls/3dvar_mult_obs.yaml, root={cost function => {cost type => 3D-Var , window begin => 2009-12-30T21:00:00Z , window length => PT6H , analysis variables => (x) , geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , background => {date => 2009-12-31T00:00:00Z , filename => qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc} , background error => {covariance model => QgError , horizontal_length_scale => 2.2e+06 , maximum_condition_number => 1e+06 , standard_deviation => 1.8e+07 , vertical_length_scale => 15000} , observations => {observers => ({obs error => {covariance model => diagonal} , obs operator => {obs type => Stream} , obs space => {obsdatain => {engine => {obsfile => qg3Dvar/output/exp_mult_obs/obs/truth_100obs.obs3d.nc}} , obsdataout => {engine => {obsfile => qg3Dvar/output/exp_mult_obs/obs

<br><br> You can see the results, after thinking about your expectations, via:

In [40]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotwind --output $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_mult_obs_increment --title "analysis increment"

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
 - fieldmax: None
 - plotObsLocations: None
 - plotObsValues: None
 - plothofx: False
 - plotwind: True
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_mult_obs_increment
 - title: analysis increment
Run script
['/home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc']
Variable x, RMSD = 27865044.207286935
data range:(-71761122.00713022,59223373.15590698), plot range: (-100000000.0,100000000.0) 
 -> plot produced: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_mult_obs_increment_x_diff.jpg
Variable q, RMSD = 2.5753222646457435e-05
data range:(-0.00017649057985359706,0.00016766328555155188), plot range: (-0.0002,0.0002) 


<br>To view these plots, navigate in the directory tree on the left to **shared->EDU->qg3Dvar->output->exp_mult_obs->plots**<br><br>
A few questions to get you going:
- Can you identify the positions of the observations? If not, why is that so difficult?
- Do you expect the increments to be larger or smaller, or similar to the default experiment? Why?

You can also look at the analysis error via:

In [41]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --output $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_mult_obs_analysis_error --title "analysis error" 

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared/EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
 - fieldmax: None
 - plotObsLocations: None
 - plotObsValues: None
 - plothofx: False
 - plotwind: False
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_mult_obs_analysis_error
 - title: analysis error
Run script
['/home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc']
Variable x, RMSD = 17845592.371394493
data range:(-69562360.86560819,63130631.800536394), plot range: (-100000000.0,100000000.0) 
 -> plot produced: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_mult_obs_analysis_error_x_diff.jpg
Variable q, RMSD = 6.361787932025602e-05
data range:(-0.00036169651009427406,0.00028429120635015104), plot range: (-0.0002,0.0002) 
 

We previously plotted the background errors in the `0.qg_start_tutorial.ipynb` tutorial. Let's copy those images to the exp_mult_obs/plots directory, where we just produced analysis error plots. Then, open the background and analysis error plots and compare them, located at  `qg3Dvar/output/exp_mult_obs/plots` (navigate using the directory tree at left)

In [ ]:
cd $JEDI_EDU/plots_scripts
cp ./qgstart/output/plots/bkgf_error*jpg ./qg3Dvar/output/exp_mult_obs/plots

- Is the analysis error indeed smaller than the background error?
- At all locations? If not, is there any pattern to areas where the error is not smaller?

### Observation-space diagnostics 

To better diagnose these questions and help judge whether the DA performed as expected, it is helpful to perform **observation-space diagnostics and statistics**. For this task, we will run the **qg_hofx3d.x** program. This program computes the hofx of a given model input - either the analysis or background state - and outputs data in an observation file with the hofx data listed at all observation locations. The yaml for this is given by **hofx_mult_obs.yaml** (pasted below). The inputs are the background state and the observations file generated by the truth. The output of this program is another observations file, but this time containing hofx for the background state 
`(hofx_bkgd_100obs.obs3d.nc)`. 

```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: 2009-12-31T00:00:00Z
  filename: qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
window begin: 2009-12-30T21:00:00Z
window length: PT6H
observations:
  get values:
    variable change:
      input variables: []
      output variables: []
  observers:
  - obs operator:
      obs type: Stream                             # The observation type is the stream function
    obs space:
      obsdatain:
        engine:
          obsfile: qg3Dvar/output/exp_mult_obs/obs/truth_100obs.obs3d.nc
      obsdataout:
        engine:
          obsfile: qg3Dvar/output/exp_mult_obs/obs/hofx_bkgd_100obs.obs3d.nc
      obs type: Stream
    obs operator:
      obs type: Stream
    get values:
      interpolation type: default_1
```

In additrion to the background state, we want to produce a yaml for the **analysis** as will, which will compute the hofx at observation locations for the analysis. This essentially computes then the **fit of observations to the analysis**, i.e. how closely the analysis matches the assimilated observations after DA.

To do so, first make a copy of the above **hofx_mult_obs.yaml**

In [42]:
cp $JEDI_EDU/qg3Dvar/yamls/hofx_mult_obs.yaml $JEDI_EDU/qg3Dvar/yamls/hofx_mult_obs_an.yaml 
ls $JEDI_EDU/qg3Dvar/yamls/hofx_mult_obs_an.yaml 

/home/nonroot/shared/EDU/qg3Dvar/yamls/hofx_mult_obs_an.yaml


Next, optn the **hofx_mult_obs_an.yaml** and edit two places in the file:
- Replace background (`bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc`) with the analysis (`da/3dvar.an.2009-12-31T00:00:00Z.nc`) for **filename**
- Change the **outfile** under **obsdataout** to indicate it is the hofx for the analysis - i.e. rename `hofx_bkgd_100obs.obs3d.nc` with `hofx_an_100obs.obs3d.nc`

Now we can run the hofx program for both the background and analysis:

In [43]:
cd $JEDI_EDU/
$JEDI_BIN/qg_hofx3d.x ./qg3Dvar/yamls/hofx_mult_obs.yaml

Configuration input file is: ./qg3Dvar/yamls/hofx_mult_obs.yaml
OOPS Starting 2026-02-12 18:25:19 (UTC+0000)
Full configuration is:YAMLConfiguration[path=./qg3Dvar/yamls/hofx_mult_obs.yaml, root={geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , state => {date => 2009-12-31T00:00:00Z , filename => qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc} , window begin => 2009-12-30T21:00:00Z , window length => PT6H , observations => {get values => {variable change => {input variables => () , output variables => ()}} , observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {engine => {obsfile => qg3Dvar/output/exp_mult_obs/obs/truth_100obs.obs3d.nc}} , obsdataout => {engine => {obsfile => qg3Dvar/output/exp_mult_obs/obs/hofx_bkgd_100obs.obs3d.nc}} , obs type => Stream} , get values => {interpolation type => default_1}})}}]
OOPS_STATS ObjectCountHelper started.
OOPS_STATS Run start                                - Runtime:      0.03 sec,  Me

In [44]:
cd $JEDI_EDU/
$JEDI_BIN/qg_hofx3d.x ./qg3Dvar/yamls/hofx_mult_obs_an.yaml

Configuration input file is: ./qg3Dvar/yamls/hofx_mult_obs_an.yaml
OOPS Starting 2026-02-12 18:25:25 (UTC+0000)
Full configuration is:YAMLConfiguration[path=./qg3Dvar/yamls/hofx_mult_obs_an.yaml, root={geometry => {nx => 40 , ny => 20 , depths => (4000,6000)} , state => {date => 2009-12-31T00:00:00Z , filename => qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc} , window begin => 2009-12-30T21:00:00Z , window length => PT6H , observations => {get values => {variable change => {input variables => () , output variables => ()}} , observers => ({obs operator => {obs type => Stream} , obs space => {obsdatain => {engine => {obsfile => qg3Dvar/output/exp_mult_obs/obs/truth_100obs.obs3d.nc}} , obsdataout => {engine => {obsfile => qg3Dvar/output/exp_mult_obs/obs/hofx_an_100obs.obs3d.nc}} , obs type => Stream} , get values => {interpolation type => default_1}})}}]
OOPS_STATS ObjectCountHelper started.
OOPS_STATS Run start                                - Runtime:      0.03 sec,  M

As with previous obs files generated, we can use the plot.py program to extract the observations in text files:

In [45]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_bkgd_100obs.obs3d.nc\
        --output $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/qg_multobs_bkgd
cat $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/qg_multobs_bkgd.txt

python ./plot.py qg obs $JEDI_EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_an_100obs.obs3d.nc\
        --output $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/qg_multobs_an
cat $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/qg_multobs_an.txt

Parameters:
 - model: qg
 - diagnostic: obs
 - filepath: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_bkgd_100obs.obs3d.nc
 - output: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/qg_multobs_bkgd
Run script
 -> Observations values written in /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/qg_multobs_bkgd.txt
# location / value / hofx
[ -29.87208056    3.63767342 3266.44902118] / [3115105.01660962] / [-8980454.95365635]
[ 178.98653093    8.23197272 5786.33931931] / [9341113.13679803] / [16739348.1077763]
[  79.31681614   59.17619073 5270.58105916] / [-1.75884984e+08] / [-1.99086365e+08]
[155.72065003  78.38089478  90.07995483] / [-93802685.12679093] / [-69996005.69573534]
[-179.95882281   18.17969161 8859.42095192] / [-26903573.09265] / [-47136716.06133686]
[-133.8751988    77.90727735 7090.42522358] / [-3.60603479e+08] / [-3.85352564e+08]
[ -71.16027561   22.64921372 3572.69760454] / [-78510060.90002471] / [-11585538.03830298]
[ 179.65458554   2

Statistics can be performed with these data, such as computation of Root-Mean-Squared-Error (RMSE). They can also be powerful diagnostics for evaluating whether the DA performed as expected and visualizing how well an analysis performs, as we will demonstrate now.

We can also revisit previous plots for background error and analysis error. The plot.py program below uses the `--plotObsValues` option to <u>include observation locations and the `y-hofx` values  for background and analysis</u>. These quantities represent the **observation innovation** (y-h(Xb), i.e. how much error must be corrected in the background compared to observations) and the **analysis fit** (y-h(Xa), i.e. the error in the final analysis compared to observations).


In [46]:
    cd $JEDI_EDU/plots_scripts
    python ./plot.py qg fields \
            $JEDI_EDU/qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
            $JEDI_EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
            --plotObsValues $JEDI_EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_bkgd_100obs.obs3d.nc \
            --output $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_bkgdErr_withobs --title "Background error"

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
 - basefilepath: /home/nonroot/shared/EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
 - fieldmax: None
 - plotObsLocations: None
 - plotObsValues: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_bkgd_100obs.obs3d.nc
 - plothofx: False
 - plotwind: False
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_bkgdErr_withobs
 - title: Background error
Run script
['/home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc']
lon,lat,val of first ob = -29.872080562636263,3.6376734229618757,12095559.970265962
Variable x, RMSD = 32505586.64522618
data range:(-81306687.9098106,102677469.32181332), plot range: (-100000000.0,100000000.0) 
 -> plot produced: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_bkgdErr_

In [47]:
    cd $JEDI_EDU/plots_scripts
    python ./plot.py qg fields \
            $JEDI_EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc \
            $JEDI_EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
            --plotObsValues $JEDI_EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_an_100obs.obs3d.nc \
            --output $JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_anErr_withobs --title "Analysis error"

Parameters:
 - model: qg
 - diagnostic: fields
 - filepath: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc
 - basefilepath: /home/nonroot/shared/EDU/qg3Dvar/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
 - fieldmax: None
 - plotObsLocations: None
 - plotObsValues: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/obs/hofx_an_100obs.obs3d.nc
 - plothofx: False
 - plotwind: False
 - plotstream: False
 - gif: None
 - output: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_anErr_withobs
 - title: Analysis error
Run script
['/home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/da/3dvar.an.2009-12-31T00:00:00Z.nc']
lon,lat,val of first ob = -29.872080562636263,3.6376734229618757,-2525200.558360869
Variable x, RMSD = 17845592.371394493
data range:(-69562360.86560819,63130631.800536394), plot range: (-100000000.0,100000000.0) 
 -> plot produced: /home/nonroot/shared/EDU/qg3Dvar/output/exp_mult_obs/plots/3dvar_anErr_withobs_x_di

To reiterate, these observation dots are colored with the values of `y-h(x)` for the background and analysis, to give a sense of the errors before and after assimilation, how the DA should correct nearby model points, and the result after running 3dvar. With these new plots to help, can you answer the following questions a bit more clearly?
- How do you interpret each plot?
- Is the analysis error indeed smaller than the background error?
- In every grid point? For all observations? Or on average?
- Are there areas where there is more anlaysis error? If so, what patterns do you notice about these areas?
  

On the question of if it's all grid points with lower analysis errors or just on average, there is an interesting piece of theory in Kalman filtering that carries over directly to a 3DVar if the observation operator $H$ is linear. From the Kalman filter equations at the beginning of this 3DVar exercise we can derive an update equation for the model covariance matrix, as:
<center> $P = (I-KH)B$ </center>

By substituting the expression for Kalman gain K in we find:
<center> $P = B - BH^T(HBH^T + R)^{-1} HB \;\;\;\;\;$ so that $ \;\;\;\;\;Tr(P) = Tr(B) - Tr(BH^T(HBH^T + R)^{-1} HB)$  </center> 

All three matrices are symmetric positive semi definite, meaning that their eigenvalues are non-negative, and the trace is the sum of the eigenvalues. We thus see that $Tr(P) \leq Tr(B)$. The trace of a covariance matrix is the sum of the diagonal elements, so the sum of the variances of all variables in the state vector. Hence, we find that **the sum of the error variances of the analysis will be lower than that of the background error variances.** 

This, then, shows that the **average absolute value of the analysis errors is expected to be smaller than that of the background errors**. So, while not all locations have lower analysis errors than background errors, in an **average** sense it is true as shown empirically and through the DA equations directly. 

This feature of the Kalman filter, and  the 3DVar for linear observation operators, can be traced back to its derivation: the Kalman filter can be derived as the minimum variance estimator for linear problems.

This experiment should have

- Further increased your skills in using and changing yaml files
- Given you a even better understanding of the covariances 3DVar (and in data assimilation in general), leading to an almost full understanding of a 3DVar.